# ANPR-India — End-to-end Training (Kaggle, T4 GPU)

This notebook runs the full v1.1 pipeline:
1. Install deps
2. Download pretrained weights (YOLO26, Real-ESRGAN, PARSeq)
3. Generate 150k synthetic Indian plates
4. Convert CCPD to YOLO pose format (detector pretrain data)
5. Train detector (Stage 1: pretrain → Stage 2: fine-tune)
6. Train LPRNet recognizer (Stage 1: synthetic pretrain → Stage 2: real fine-tune)
7. Export to ONNX
8. Run smoke test on a sample image

**Architecture (v1.1):** YOLO26s-pose → rectify → LPRNet (fast) / PARSeq (heavy) → grammar.
See `docs/REVIEW_AND_RECOMMENDATIONS.md` for the full spec.

**Setup:**
- Add this project as a Kaggle Dataset (upload `anpr-india-kaggle.zip`)
- Set accelerator: **GPU T4 x2** (or P100)
- Set environment: Internet on (for pip installs + HF Hub downloads)

In [ ]:
# === Setup ===
import sys, os
from pathlib import Path

# Path to the uploaded dataset (adjust if your dataset slug differs)
PROJECT_DIR = Path('/kaggle/input/anpr-india-kaggle')
sys.path.insert(0, str(PROJECT_DIR / 'src'))
sys.path.insert(0, str(PROJECT_DIR))
os.environ['KAGGLE_INPUT'] = str(PROJECT_DIR)

# Working directories (writable on Kaggle)
WEIGHTS_DIR = Path('/kaggle/working/weights')
DATA_DIR = Path('/kaggle/working/data')
OUTPUT_DIR = Path('/kaggle/working/output')
for d in (WEIGHTS_DIR, DATA_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'Project: {PROJECT_DIR}')
print(f'Weights: {WEIGHTS_DIR}')
print(f'Data: {DATA_DIR}')
print(f'Output: {OUTPUT_DIR}')
print(f'Python: {sys.version}')

In [ ]:
# === Install dependencies ===
# Kaggle has torch, numpy, opencv, pyyaml preinstalled.
# We need ultralytics (YOLO26), onnxruntime, supervision, etc.
!pip install -q ultralytics>=8.3.0 onnxruntime-gpu supervision huggingface_hub pydantic fastapi python-multipart
!pip install -q ruff mypy pytest
# Optional: paddleocr for the Wk1-2 baseline
# !pip install -q paddlepaddle-gpu paddleocr

import torch, numpy as np, cv2
print(f'torch: {torch.__version__}, cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# === Run unit tests (sanity check) ===
!cd {PROJECT_DIR} && PYTHONPATH=src python -m pytest tests/ -v --tb=short 2>&1 | tail -20

In [ ]:
# === Smoke test: LPRNet architecture ===
# Verifies the model constructs + forward pass + decode work.
!cd {PROJECT_DIR} && PYTHONPATH=src python -m anpr.models.lprnet

In [ ]:
# === Download pretrained weights ===
from scripts.download_weights import download_all
download_all(WEIGHTS_DIR)
!ls -lh {WEIGHTS_DIR}

In [ ]:
# === Generate synthetic Indian plates (150k strips, ~1 GPU-hour) ===
from scripts.make_synthetic_plates import generate
generate(150_000, DATA_DIR / 'synthetic_indian_plates', seed=42)
!ls {DATA_DIR / 'synthetic_indian_plates' / 'images'} | head -3
!wc -l {DATA_DIR / 'synthetic_indian_plates' / 'labels.csv'}

In [ ]:
# === (Optional) Convert CCPD to YOLO pose format ===
# Only run if you've added CCPD as a Kaggle dataset.
ccpd_input = Path('/kaggle/input/ccpd2019/ccpd_base')
if ccpd_input.exists():
    from scripts.prepare_ccpd import process
    process(ccpd_input, DATA_DIR / 'ccpd_yolo', sample_n=20_000)
else:
    print(f'CCPD not found at {ccpd_input} — skipping detector pretrain data prep.')
    print('Add CCPD as a Kaggle Dataset to enable detector pretraining.')

In [ ]:
# === (Optional) Download Roboflow datasets ===
# Requires ROBOFLOW_API_KEY Kaggle secret. Add via Add-ons → Secrets.
import os
if 'ROBOFLOW_API_KEY' in os.environ:
    from scripts.prepare_roboflow import download_one, combine_into_unified, ROBOFLOW_DATASETS
    for spec in ROBOFLOW_DATASETS:
        download_one(spec, DATA_DIR / 'roboflow')
    combine_into_unified(DATA_DIR / 'roboflow', ROBOFLOW_DATASETS)
else:
    print('ROBOFLOW_API_KEY not set — skipping Roboflow downloads.')
    print('Get a free key at https://roboflow.com → Settings → API')
    print('Then add it as a Kaggle Secret and re-run this cell.')

In [ ]:
# === Train LPRNet recognizer (Stage 1: synthetic pretrain) ===
# This runs entirely on synthetic data — no external dataset required.
from anpr.models.lprnet_trainer import train, TrainConfig

cfg = TrainConfig(
    pretrain_data=DATA_DIR / 'synthetic_indian_plates',
    finetune_data=None,   # skip fine-tune if no real crops yet
    pretrain_epochs=10,
    pretrain_batch_size=256,
    device='cuda',
    output_dir=OUTPUT_DIR / 'lprnet',
    num_workers=4,
)
best_path = train(cfg)
print(f'Best LPRNet checkpoint: {best_path}')

In [ ]:
# === Train detector (Stage 1: pretrain on CCPD+Roboflow, Stage 2: fine-tune Indian) ===
# Only runs if you have detector training data prepared.
import yaml
detect_cfg_path = PROJECT_DIR / 'configs' / 'detect.yaml'
with open(detect_cfg_path) as f:
    detect_cfg = yaml.safe_load(f)

# Check if pretrain data exists
pretrain_data_path = DATA_DIR / 'ccpd_yolo'
if pretrain_data_path.exists():
    from ultralytics import YOLO
    base_weights = str(WEIGHTS_DIR / 'yolo26s-pose.pt')
    if not Path(base_weights).exists():
        base_weights = str(WEIGHTS_DIR / 'yolo11s-pose.pt')
        print(f'YOLO26 not found, using {base_weights}')
    model = YOLO(base_weights, task='pose')
    results = model.train(
        data=str(pretrain_data_path / 'data.yaml'),
        epochs=40,
        batch=32,
        imgsz=640,
        device=0,
        project=str(OUTPUT_DIR / 'runs'),
        name='detector_pretrain',
        patience=15,
        cos_lr=True,
        fliplr=0.0, flipud=0.0,   # plates are directional — never flip
    )
    print(f'Detector training done. Best weights: {results.save_dir}')
else:
    print(f'Pretrain data not found at {pretrain_data_path}')
    print('Prepare CCPD or Roboflow datasets first (see cells above).')

In [ ]:
# === Export LPRNet to ONNX ===
import torch
from anpr.models.lprnet import LPRNet, INPUT_H, INPUT_W

model = LPRNet()
ckpt = torch.load(best_path, map_location='cpu')
model.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
model.eval()

onnx_path = WEIGHTS_DIR / 'lprnet_indian_v2.onnx'
torch.onnx.export(
    model,
    torch.randn(1, 3, INPUT_H, INPUT_W),
    str(onnx_path),
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)
print(f'Exported LPRNet ONNX: {onnx_path}')
!ls -lh {onnx_path}

In [ ]:
# === Run the v1.1 grammar tests one more time to verify everything works ===
!cd {PROJECT_DIR} && PYTHONPATH=src python -m pytest tests/test_grammar.py tests/test_lprnet.py -v 2>&1 | tail -30

## Next steps

1. **Add real Indian plate crops** — fine-tune LPRNet on them (Stage 2)
2. **Train PARSeq** on the same real crops (for the heavy-path recognizer)
3. **Run the ablation matrix** A1-A10 from `scripts/run_ablation.py`
4. **Build a HF Spaces demo** using the FastAPI app in `src/anpr/api.py`

See `docs/REVIEW_AND_RECOMMENDATIONS.md` for the full v1.1 spec.